# Урок 8.2. Регуляризованная цель: веса листьев и выигрыш разбиения

**Интерактивная версия:** `lessons/lesson_8_2/web/index.html`

1. Первая итерация вручную на игрушечных данных.
2. Качество структуры: формула против точного значения цели.
3. Сверка с XGBoost: `gain` (без ½), `cover` (= H), значения листьев, параметр `gamma`.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "python" / "gbcourse").is_dir())
sys.path.insert(0, str(ROOT / "shared" / "python"))

import numpy as np
import pandas as pd
from gbcourse import datasets, GBRegressor, RegressionTree

## 1. Первая итерация вручную

In [2]:
X, y = datasets.toy_regression()
lam = 1.0
g = y.mean() - y
h = np.ones_like(g)
G, H = g.sum(), h.sum()
S = lambda G, H: G * G / (H + lam)
rows = []
for k in range(1, len(y)):
    GL, HL = g[:k].sum(), h[:k].sum()
    GR, HR = G - GL, H - HL
    rows.append({"порог": (X[k - 1, 0] + X[k, 0]) / 2, "G_L": GL, "H_L": HL, "G_R": GR, "H_R": HR,
                 "w_L": -GL / (HL + lam), "w_R": -GR / (HR + lam), "gain": 0.5 * (S(GL, HL) + S(GR, HR) - S(G, H))})
pd.DataFrame(rows).round(3)

,порог,G_L,H_L,G_R,H_R,w_L,w_R,gain
0,1.5,4.0,1.0,-4.0,5.0,-2.000,0.667,5.333
1,2.5,6.0,2.0,-6.0,4.0,-2.000,1.200,9.600
2,3.5,9.0,3.0,-9.0,3.0,-2.250,2.250,20.250
3,4.5,8.0,4.0,-8.0,2.0,-1.600,2.667,17.067
4,5.5,5.0,5.0,-5.0,1.0,-0.833,2.500,8.333


In [3]:
t = RegressionTree(max_depth=1, reg_lambda=lam).fit(X, g)
print(t.describe())

├─ если x0 ≤ 3.500  (n = 6, выигрыш = 20.250)
│   └─ лист: значение = -2.250 (n = 3)
├─ иначе (x0 > 3.500)
│   └─ лист: значение = 2.250 (n = 3)


## 2. Качество структуры

Для квадратичной потери приближение второго порядка точное, поэтому $-\tfrac12\sum G_j^2/(H_j+\lambda)$ должно
совпасть с изменением цели $\sum \tfrac12 (y - F - w)^2 + \tfrac{\lambda}{2}\sum w_j^2 - \sum \tfrac12 (y - F)^2$.

In [4]:
Xf, yf = datasets.friedman1(n=300, noise=1.0, seed=80)
F = np.full(len(yf), yf.mean())
gf = F - yf
tree = RegressionTree(max_depth=3, reg_lambda=lam).fit(Xf, gf)
w = tree.predict(Xf)
score = -0.5 * sum(lf.G**2 / (lf.H + lam) for lf in tree.leaves)
exact = 0.5 * np.sum((yf - F - w) ** 2) + 0.5 * lam * sum(lf.value**2 for lf in tree.leaves) - 0.5 * np.sum((yf - F) ** 2)
print(f"качество структуры по формуле: {score:.6f}\nточное изменение цели:          {exact:.6f}")

качество структуры по формуле: -2025.748128
точное изменение цели:          -2025.748128


## 3. Сверка с XGBoost

In [5]:
import xgboost as xgb

ref = xgb.XGBRegressor(n_estimators=1, learning_rate=1.0, max_depth=3, reg_lambda=lam, min_child_weight=0,
                       tree_method="exact", base_score=float(yf.mean())).fit(Xf, yf)
df = ref.get_booster().trees_to_dataframe()
splits = df[df.Feature != "Leaf"].sort_values("Gain", ascending=False)
ours_gains = sorted((nd.gain for nd in tree.nodes if not nd.is_leaf), reverse=True)
comp = pd.DataFrame({"XGBoost gain": splits.Gain.values, "gbcourse gain": ours_gains})
comp["отношение"] = comp["XGBoost gain"] / comp["gbcourse gain"]
comp.round(4)

,XGBoost gain,gbcourse gain,отношение
0,1474.1877,737.0939,2.0
1,798.2794,399.1397,2.0
2,569.3826,284.6913,2.0
3,482.9879,241.4940,2.0
4,455.6873,227.8436,2.0
5,138.7282,69.3641,2.0
6,132.243,66.1216,2.0


In [6]:
covers = sorted(df.Cover.values)
ours_H = sorted(nd.H for nd in tree.nodes)
print("cover == H во всех узлах:", np.allclose(covers, ours_H))
leaf_xgb = sorted(df[df.Feature == "Leaf"].Gain.values)
leaf_ours = sorted(lf.value for lf in tree.leaves)
print(f"наибольшее расхождение значений листьев: {np.abs(np.array(leaf_xgb) - np.array(leaf_ours)).max():.1e}")

cover == H во всех узлах: True
наибольшее расхождение значений листьев: 1.8e-07


### Параметр gamma: множитель 2

Возьмём γ между выигрышами двух самых слабых разбиений нашего дерева. В XGBoost тот же эффект даёт `gamma = 2γ`.
Небольшое расхождение возможно и тогда: XGBoost сначала растит дерево, а потом срезает слабые разбиения снизу вверх,
`gbcourse` отказывается от них сразу. Здесь γ отсекает только нижние разбиения, и результаты совпадают.

In [7]:
weak = sorted(nd.gain for nd in tree.nodes if not nd.is_leaf)
gamma = (weak[0] + weak[1]) / 2  # γ в масштабе статьи: разбиение остаётся, если gain_статьи > γ
ours = GBRegressor(mode="newton", n_estimators=10, learning_rate=0.3, max_depth=3, reg_lambda=lam, gamma=gamma).fit(Xf, yf)
for label, gx in (("gamma = γ", gamma), ("gamma = 2γ", 2 * gamma)):
    m = xgb.XGBRegressor(n_estimators=10, learning_rate=0.3, max_depth=3, reg_lambda=lam, gamma=gx, min_child_weight=0,
                         tree_method="exact", base_score=float(yf.mean())).fit(Xf, yf)
    print(f"XGBoost {label:11s}: наибольшее расхождение с gbcourse {np.abs(m.predict(Xf) - ours.predict(Xf)).max():.1e}")

XGBoost gamma = γ  : наибольшее расхождение с gbcourse 3.0e+00
XGBoost gamma = 2γ : наибольшее расхождение с gbcourse 4.9e-06


## Упражнения

1. Повторите таблицу раздела 1 для λ = 0 и λ = 10. Меняется ли лучший порог?
2. Логистическая потеря, 4 объекта с $p = 0.5$: $y = (0, 0, 1, 1)$, $x = (1, 2, 3, 4)$. Посчитайте вручную
   выигрыш порога 2.5 при λ = 0 и λ = 1.

Решения: `python lessons/lesson_8_2/exercises/solutions.py`.